# CMI Sensor Behavior Replica — Strong Teacher Submission

Teacher solution

It adds:

- subject-specific sensor correction
- handedness normalization
- right-hand model
- left-hand model
- TTA-style probability averaging
- physics-style IMU/rotation features
- normal Kaggle `submission.csv`

No evaluation API is used.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option("display.max_columns", 250)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

print("LightGBM available:", HAS_LGBM)

LightGBM available: True


In [2]:
INPUT_DIR = Path("/kaggle/input/competitions/comp-sensor-timeseries")

train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")
train_demo = pd.read_csv(INPUT_DIR / "train_demographics.csv")
test_demo = pd.read_csv(INPUT_DIR / "test_demographics.csv")
sample_submission = pd.read_csv(INPUT_DIR / "sample_submission.csv")

print("train:", train.shape)
print("test:", test.shape)
display(train.head())
display(train_demo.head())

train: (459122, 16)
test: (115823, 15)


,row_id,sequence_type,sequence_id,sequence_counter,subject,orientation,behavior,phase,gesture,acc_x,acc_y,acc_z,rot_w,rot_x,rot_y,rot_z
0,SEQ_000007_000000,Target,SEQ_000007,0,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.683594,6.214844,3.355469,0.134399,-0.355164,-0.447327,-0.809753
1,SEQ_000007_000001,Target,SEQ_000007,1,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.949219,6.214844,3.125000,0.143494,-0.340271,-0.428650,-0.824524
2,SEQ_000007_000002,Target,SEQ_000007,2,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.722656,5.410156,5.421875,0.219055,-0.274231,-0.356934,-0.865662
3,SEQ_000007_000003,Target,SEQ_000007,3,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.601562,3.531250,6.457031,0.297546,-0.264160,-0.238159,-0.885986
4,SEQ_000007_000004,Target,SEQ_000007,4,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.566406,0.277344,9.632812,0.333557,-0.218628,-0.063538,-0.914856


,subject,adult_child,age,sex,handedness,height_cm,shoulder_to_wrist_cm,elbow_to_wrist_cm
0,SUBJ_000206,1,41,1,1,172.0,50,25.0
1,SUBJ_001430,0,11,0,1,167.0,51,27.0
2,SUBJ_002923,1,28,1,0,164.0,54,26.0
3,SUBJ_003328,1,33,1,1,171.0,52,25.0
4,SUBJ_008728,0,15,1,1,175.0,51,24.0


## Metric

In [3]:
TARGET_GESTURES = [
    "Above ear - pull hair",
    "Cheek - pinch skin",
    "Eyebrow - pull hair",
    "Eyelash - pull hair",
    "Forehead - pull hairline",
    "Forehead - scratch",
    "Neck - pinch skin",
    "Neck - scratch",
]

def hierarchical_f1_score(y_true, y_pred):
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)

    y_true_binary = y_true.isin(TARGET_GESTURES)
    y_pred_binary = y_pred.isin(TARGET_GESTURES)

    f1_binary = f1_score(
        y_true_binary,
        y_pred_binary,
        pos_label=True,
        average="binary",
        zero_division=0,
    )

    y_true_mc = y_true.apply(lambda x: x if x in TARGET_GESTURES else "non_target")
    y_pred_mc = y_pred.apply(lambda x: x if x in TARGET_GESTURES else "non_target")

    f1_macro = f1_score(
        y_true_mc,
        y_pred_mc,
        average="macro",
        zero_division=0,
    )

    return 0.5 * f1_binary + 0.5 * f1_macro

## Feature engineering helpers

In [4]:
ID_COL = "sequence_id"
TARGET_COL = "gesture"
SUBJECT_COL = "subject"

BASE_SENSOR_COLS = [
    "acc_x", "acc_y", "acc_z",
    "rot_w", "rot_x", "rot_y", "rot_z",
]

def safe_diff_fillna_zero(s):
    return s.diff().fillna(0)

def normalize_quaternion_array(q):
    q = q.astype("float64")
    norm = np.linalg.norm(q, axis=1, keepdims=True)
    norm = np.where(norm == 0, 1.0, norm)
    return q / norm

def quaternion_multiply(q1, q2):
    w1, x1, y1, z1 = q1[:, 0], q1[:, 1], q1[:, 2], q1[:, 3]
    w2, x2, y2, z2 = q2[:, 0], q2[:, 1], q2[:, 2], q2[:, 3]
    return np.column_stack([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    ])

def quaternion_conjugate(q):
    out = q.copy()
    out[:, 1:] *= -1
    return out

def add_subject_corrections(df):
    df = df.copy()

    mask = df[SUBJECT_COL].eq("SUBJ_019262")
    for c in ["acc_y", "rot_y"]:
        if c in df.columns:
            df.loc[mask, c] = -df.loc[mask, c]

    mask = df[SUBJECT_COL].eq("SUBJ_045235")
    for c in ["acc_x", "acc_y", "rot_x", "rot_y"]:
        if c in df.columns:
            df.loc[mask, c] = -df.loc[mask, c]

    return df

def apply_handedness_view(df, demographics, target_hand="right"):
    df = df.copy()
    demo_small = demographics[[SUBJECT_COL, "handedness"]].drop_duplicates()
    df = df.merge(demo_small, on=SUBJECT_COL, how="left")

    if target_hand == "right":
        should_flip = df["handedness"].eq(0)
    elif target_hand == "left":
        should_flip = df["handedness"].eq(1)
    else:
        raise ValueError("target_hand must be 'right' or 'left'")

    # Mirroring transform for simplified IMU + rotation setting.
    mirror_cols = [c for c in ["acc_x", "rot_y", "rot_z"] if c in df.columns]
    df.loc[should_flip, mirror_cols] = -df.loc[should_flip, mirror_cols]

    return df.drop(columns=["handedness"])

def add_rowwise_motion_features(df):
    df = df.copy()

    df["acc_mag"] = np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2)

    q_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]
    q = normalize_quaternion_array(df[q_cols].fillna(0).to_numpy())
    w, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]

    df["rot_angle"] = 2 * np.arccos(np.clip(np.abs(w), -1, 1))

    sinr_cosp = 2 * (w*x + y*z)
    cosr_cosp = 1 - 2 * (x*x + y*y)
    df["roll"] = np.arctan2(sinr_cosp, cosr_cosp)

    sinp = 2 * (w*y - z*x)
    df["pitch"] = np.where(
        np.abs(sinp) >= 1,
        np.sign(sinp) * np.pi / 2,
        np.arcsin(sinp)
    )

    siny_cosp = 2 * (w*z + x*y)
    cosy_cosp = 1 - 2 * (y*y + z*z)
    df["yaw"] = np.arctan2(siny_cosp, cosy_cosp)

    gravity_x = 2 * (x*z - w*y)
    gravity_y = 2 * (w*x + y*z)
    gravity_z = w*w - x*x - y*y + z*z

    df["linear_acc_x"] = df["acc_x"] - gravity_x
    df["linear_acc_y"] = df["acc_y"] - gravity_y
    df["linear_acc_z"] = df["acc_z"] - gravity_z
    df["linear_acc_mag"] = np.sqrt(
        df["linear_acc_x"]**2 + df["linear_acc_y"]**2 + df["linear_acc_z"]**2
    )

    df["acc_mag_jerk"] = df.groupby(ID_COL)["acc_mag"].transform(safe_diff_fillna_zero)
    df["linear_acc_mag_jerk"] = df.groupby(ID_COL)["linear_acc_mag"].transform(safe_diff_fillna_zero)
    df["rot_angle_vel"] = df.groupby(ID_COL)["rot_angle"].transform(safe_diff_fillna_zero)

    df[["angular_vel_x", "angular_vel_y", "angular_vel_z", "angular_distance"]] = 0.0

    # Fill angular velocity sequence by sequence.
    # This is simple and safe enough for a few thousand sequences.
    for _, idx in df.groupby(ID_COL, sort=False).groups.items():
        idx = np.asarray(list(idx))
        if len(idx) <= 1:
            continue

        q_seq = q[idx]
        q_prev = q_seq[:-1]
        q_next = q_seq[1:]
        dq = quaternion_multiply(q_next, quaternion_conjugate(q_prev))
        dq = normalize_quaternion_array(dq)

        angle = 2 * np.arccos(np.clip(np.abs(dq[:, 0]), -1, 1))
        sin_half = np.sqrt(np.maximum(1 - dq[:, 0]**2, 1e-12))
        axis = dq[:, 1:4] / sin_half[:, None]
        ang_vel = axis * angle[:, None]

        df.loc[idx[1:], ["angular_vel_x", "angular_vel_y", "angular_vel_z"]] = ang_vel
        df.loc[idx[1:], "angular_distance"] = angle

    df["angular_vel_mag"] = np.sqrt(
        df["angular_vel_x"]**2 + df["angular_vel_y"]**2 + df["angular_vel_z"]**2
    )
    df["angular_vel_mag_jerk"] = df.groupby(ID_COL)["angular_vel_mag"].transform(safe_diff_fillna_zero)

    return df

ENGINEERED_ROW_COLS = [
    "acc_mag", "acc_mag_jerk",
    "rot_angle", "rot_angle_vel",
    "linear_acc_mag", "linear_acc_mag_jerk",
    "angular_vel_x", "angular_vel_y", "angular_vel_z",
    "angular_vel_mag", "angular_vel_mag_jerk",
    "angular_distance",
    "roll", "pitch", "yaw",
]

## Sequence aggregation

In [5]:
def make_sequence_features(raw_df, demographics, target_hand):
    df = raw_df.copy()

    df = add_subject_corrections(df)
    df = apply_handedness_view(df, demographics, target_hand=target_hand)
    df = add_rowwise_motion_features(df)

    sensor_cols = [c for c in BASE_SENSOR_COLS + ENGINEERED_ROW_COLS if c in df.columns]

    grouped = df.groupby(ID_COL, sort=False)
    pieces = []

    stats = grouped[sensor_cols].agg(["mean", "std", "min", "max", "median"])
    stats.columns = [f"{c}_{s}" for c, s in stats.columns]
    pieces.append(stats)

    for qv in [0.10, 0.25, 0.75, 0.90]:
        quant = grouped[sensor_cols].quantile(qv)
        quant.columns = [f"{c}_q{int(qv*100)}" for c in quant.columns]
        pieces.append(quant)

    first_vals = grouped[sensor_cols].first()
    first_vals.columns = [f"{c}_first" for c in first_vals.columns]
    pieces.append(first_vals)

    last_vals = grouped[sensor_cols].last()
    last_vals.columns = [f"{c}_last" for c in last_vals.columns]
    pieces.append(last_vals)

    delta_vals = grouped[sensor_cols].last() - grouped[sensor_cols].first()
    delta_vals.columns = [f"{c}_delta" for c in delta_vals.columns]
    pieces.append(delta_vals)

    miss = grouped[sensor_cols].apply(lambda x: x.isna().mean())
    miss.columns = [f"{c}_missing_frac" for c in miss.columns]
    pieces.append(miss)

    seq_len = grouped.size().to_frame("sequence_length")
    pieces.append(seq_len)

    subject = grouped[SUBJECT_COL].first().to_frame(SUBJECT_COL)
    pieces.append(subject)

    return pd.concat(pieces, axis=1).reset_index()

def build_view(raw_train, raw_test, train_demo, test_demo, target_hand):
    train_seq = make_sequence_features(raw_train, train_demo, target_hand=target_hand)
    test_seq = make_sequence_features(raw_test, test_demo, target_hand=target_hand)

    labels = raw_train[[ID_COL, TARGET_COL]].drop_duplicates()
    train_seq = train_seq.merge(labels, on=ID_COL, how="left")

    train_seq = train_seq.merge(train_demo, on=SUBJECT_COL, how="left")
    test_seq = test_seq.merge(test_demo, on=SUBJECT_COL, how="left")

    return train_seq, test_seq

train_right, test_right = build_view(train, test, train_demo, test_demo, target_hand="right")
train_left, test_left = build_view(train, test, train_demo, test_demo, target_hand="left")

print("right:", train_right.shape, test_right.shape)
print("left:", train_left.shape, test_left.shape)
display(train_right.head())

right: (6530, 297) (1621, 296)
left: (6530, 297) (1621, 296)


,sequence_id,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_x_median,acc_y_mean,acc_y_std,acc_y_min,acc_y_max,acc_y_median,acc_z_mean,acc_z_std,acc_z_min,acc_z_max,acc_z_median,rot_w_mean,rot_w_std,rot_w_min,rot_w_max,rot_w_median,rot_x_mean,rot_x_std,rot_x_min,rot_x_max,rot_x_median,rot_y_mean,rot_y_std,rot_y_min,rot_y_max,rot_y_median,rot_z_mean,rot_z_std,rot_z_min,rot_z_max,rot_z_median,acc_mag_mean,acc_mag_std,acc_mag_min,acc_mag_max,acc_mag_median,acc_mag_jerk_mean,acc_mag_jerk_std,acc_mag_jerk_min,acc_mag_jerk_max,acc_mag_jerk_median,rot_angle_mean,rot_angle_std,rot_angle_min,rot_angle_max,rot_angle_median,rot_angle_vel_mean,rot_angle_vel_std,rot_angle_vel_min,rot_angle_vel_max,rot_angle_vel_median,linear_acc_mag_mean,linear_acc_mag_std,linear_acc_mag_min,linear_acc_mag_max,linear_acc_mag_median,linear_acc_mag_jerk_mean,linear_acc_mag_jerk_std,linear_acc_mag_jerk_min,linear_acc_mag_jerk_max,linear_acc_mag_jerk_median,angular_vel_x_mean,angular_vel_x_std,angular_vel_x_min,angular_vel_x_max,angular_vel_x_median,angular_vel_y_mean,angular_vel_y_std,angular_vel_y_min,angular_vel_y_max,angular_vel_y_median,angular_vel_z_mean,angular_vel_z_std,angular_vel_z_min,angular_vel_z_max,angular_vel_z_median,angular_vel_mag_mean,angular_vel_mag_std,angular_vel_mag_min,angular_vel_mag_max,angular_vel_mag_median,angular_vel_mag_jerk_mean,angular_vel_mag_jerk_std,angular_vel_mag_jerk_min,angular_vel_mag_jerk_max,angular_vel_mag_jerk_median,angular_distance_mean,angular_distance_std,angular_distance_min,angular_distance_max,angular_distance_median,roll_mean,roll_std,roll_min,roll_max,roll_median,pitch_mean,pitch_std,pitch_min,pitch_max,pitch_median,yaw_mean,yaw_std,yaw_min,yaw_max,yaw_median,acc_x_q10,acc_y_q10,acc_z_q10,rot_w_q10,rot_x_q10,rot_y_q10,rot_z_q10,acc_mag_q10,acc_mag_jerk_q10,rot_angle_q10,rot_angle_vel_q10,linear_acc_mag_q10,linear_acc_mag_jerk_q10,angular_vel_x_q10,...,angular_vel_mag_jerk_q75,angular_distance_q75,roll_q75,pitch_q75,yaw_q75,acc_x_q90,acc_y_q90,acc_z_q90,rot_w_q90,rot_x_q90,rot_y_q90,rot_z_q90,acc_mag_q90,acc_mag_jerk_q90,rot_angle_q90,rot_angle_vel_q90,linear_acc_mag_q90,linear_acc_mag_jerk_q90,angular_vel_x_q90,angular_vel_y_q90,angular_vel_z_q90,angular_vel_mag_q90,angular_vel_mag_jerk_q90,angular_distance_q90,roll_q90,pitch_q90,yaw_q90,acc_x_first,acc_y_first,acc_z_first,rot_w_first,rot_x_first,rot_y_first,rot_z_first,acc_mag_first,acc_mag_jerk_first,rot_angle_first,rot_angle_vel_first,linear_acc_mag_first,linear_acc_mag_jerk_first,angular_vel_x_first,angular_vel_y_first,angular_vel_z_first,angular_vel_mag_first,angular_vel_mag_jerk_first,angular_distance_first,roll_first,pitch_first,yaw_first,acc_x_last,acc_y_last,acc_z_last,rot_w_last,rot_x_last,rot_y_last,rot_z_last,acc_mag_last,acc_mag_jerk_last,rot_angle_last,rot_angle_vel_last,linear_acc_mag_last,linear_acc_mag_jerk_last,angular_vel_x_last,angular_vel_y_last,angular_vel_z_last,angular_vel_mag_last,angular_vel_mag_jerk_last,angular_distance_last,roll_last,pitch_last,yaw_last,acc_x_delta,acc_y_delta,acc_z_delta,rot_w_delta,rot_x_delta,rot_y_delta,rot_z_delta,acc_mag_delta,acc_mag_jerk_delta,rot_angle_delta,rot_angle_vel_delta,linear_acc_mag_delta,linear_acc_mag_jerk_delta,angular_vel_x_delta,angular_vel_y_delta,angular_vel_z_delta,angular_vel_mag_delta,angular_vel_mag_jerk_delta,angular_distance_delta,roll_delta,pitch_delta,yaw_delta,acc_x_missing_frac,acc_y_missing_frac,acc_z_missing_frac,rot_w_missing_frac,rot_x_missing_frac,rot_y_missing_frac,rot_z_missing_frac,acc_mag_missing_frac,acc_mag_jerk_missing_frac,rot_angle_missing_frac,rot_angle_vel_missing_frac,linear_acc_mag_missing_frac,linear_acc_mag_jerk_missing_frac,angular_vel_x_missing_frac,angular_vel_y_missing_frac,angular_vel_z_missing_frac,angular_vel_mag_missing_frac,angular_vel_mag_jerk_missing_frac,angular_distance_missing_frac,roll_missing_frac,pitch_missing_frac,yaw_missing_frac,sequence_length,subject,gesture,adult_child,age,sex,handedness,height_cm,shoulder_to_wrist_cm,elbow_to_wrist_cm


## Prepare model matrices

In [6]:
excluded = {ID_COL, TARGET_COL, SUBJECT_COL}

feature_cols = [
    c for c in train_right.columns
    if c not in excluded and pd.api.types.is_numeric_dtype(train_right[c])
]

feature_cols = [
    c for c in feature_cols
    if c in train_left.columns and c in test_right.columns and c in test_left.columns
]

X_right = train_right[feature_cols].copy()
X_left = train_left[feature_cols].copy()
X_test_right = test_right[feature_cols].copy()
X_test_left = test_left[feature_cols].copy()

y_text = train_right[TARGET_COL].copy()
groups = train_right[SUBJECT_COL].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

print("features:", len(feature_cols))
print("X_right:", X_right.shape)
print("X_left:", X_left.shape)
print("X_test_right:", X_test_right.shape)
print("X_test_left:", X_test_left.shape)
print("classes:", list(label_encoder.classes_))

assert X_right.select_dtypes(include=["object", "category"]).shape[1] == 0
assert X_left.select_dtypes(include=["object", "category"]).shape[1] == 0

features: 294
X_right: (6530, 294)
X_left: (6530, 294)
X_test_right: (1621, 294)
X_test_left: (1621, 294)
classes: ['Above ear - pull hair', 'Cheek - pinch skin', 'Drink from bottle/cup', 'Eyebrow - pull hair', 'Eyelash - pull hair', 'Feel around in tray and pull out an object', 'Forehead - pull hairline', 'Forehead - scratch', 'Glasses on/off', 'Neck - pinch skin', 'Neck - scratch', 'Pinch knee/leg skin', 'Pull air toward your face', 'Scratch knee/leg skin', 'Text on phone', 'Wave hello', 'Write name in air', 'Write name on leg']


## Model

In [7]:
def make_model(seed=2026):
    if HAS_LGBM:
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                n_estimators=900,
                learning_rate=0.025,
                num_leaves=63,
                min_child_samples=20,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                random_state=seed,
                objective="multiclass",
                verbosity=-1,

                # GPU settings
                device="gpu",
                gpu_use_dp=False,
                max_bin=63,
            ))
        ])
    else:
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.04,
                max_iter=500,
                l2_regularization=0.05,
                random_state=seed,
            ))
        ])

def predict_proba_aligned(model, X):
    proba = model.predict_proba(X)
    classes = model.named_steps["model"].classes_

    aligned = np.zeros((len(X), len(label_encoder.classes_)), dtype=float)
    for j, cls in enumerate(classes):
        aligned[:, int(cls)] = proba[:, j]
    return aligned

## CV with handedness TTA

## Final submission

In [8]:
final_right = make_model(seed=2026)
final_left = make_model(seed=3026)

final_right.fit(X_right, y)
final_left.fit(X_left, y)

p_test_right = predict_proba_aligned(final_right, X_test_right)
p_test_left = predict_proba_aligned(final_left, X_test_left)

p_test = 0.5 * p_test_right + 0.5 * p_test_left
test_pred = label_encoder.inverse_transform(p_test.argmax(axis=1))

pred_df = pd.DataFrame({
    ID_COL: test_right[ID_COL],
    TARGET_COL: test_pred,
})

submission = sample_submission[[ID_COL]].merge(pred_df, on=ID_COL, how="left")

assert list(submission.columns) == [ID_COL, TARGET_COL]
assert len(submission) == len(sample_submission)
assert submission[TARGET_COL].isna().sum() == 0

submission.to_csv("submission.csv", index=False)

display(submission.head())
print("Saved submission.csv")
print(submission[TARGET_COL].value_counts())

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature 

,sequence_id,gesture
0,SEQ_000016,Write name on leg
1,SEQ_000018,Forehead - scratch
2,SEQ_000166,Eyebrow - pull hair
3,SEQ_000169,Above ear - pull hair
4,SEQ_000174,Neck - scratch


Saved submission.csv
gesture
Forehead - scratch                            166
Forehead - pull hairline                      164
Text on phone                                 152
Above ear - pull hair                         145
Eyelash - pull hair                           131
Neck - scratch                                116
Cheek - pinch skin                            106
Neck - pinch skin                             106
Write name in air                              95
Eyebrow - pull hair                            90
Pull air toward your face                      83
Wave hello                                     80
Write name on leg                              45
Glasses on/off                                 32
Drink from bottle/cup                          30
Scratch knee/leg skin                          28
Feel around in tray and pull out an object     27
Pinch knee/leg skin                            25
Name: count, dtype: int64
